In [2]:
## -*- coding: utf-8 -*-
"""
Sun-Synchronous Orbit with 1 km Swath Coverage
Simulates a 474.066 km sun-synchronous orbit (~97.17°).
Generates 200 of ground tracks with 1 km swath polygons (specified on line 84)
and saves them to a KML file.

Author: drai
"""

import numpy as np
import simplekml
from math import pi, sqrt

# Orbital parameters
Re = 6371.0               # Earth radius [km]
mu = 398600.4418          # Earth GM [km^3/s^2]
J2 = 1.08263e-3
omega_earth = 2*pi / 86164.0905   # Earth rotation rate [rad/s]

alt_km = 474.066          # altitude [km]
a = Re + alt_km           # semi-major axis [km]

# Sun-synchronous inclination for 500 km
n = sqrt(mu / a**3)       # mean motion [rad/s]
target_dOmega = 2*pi / (365.2422*86400.0)  # rad/s
cos_i = -2*target_dOmega / (3*n*J2*(Re/a)**2)
inc_deg = np.degrees(np.arccos(cos_i))
print(f"Sun-sync inclination at {alt_km:.0f} km: {inc_deg:.2f}°")

period = 2*pi / n
orbits_per_day = 86400.0 / period
print(f"Period: {period/60:.2f} min, {orbits_per_day:.2f} orbits/day")

# Functions
def latlon_from_orbit(incl_deg, t, period, orbit_index):

# Compute ground track lat/lon for one orbit
    incl = np.radians(incl_deg)
    theta = 2*pi*(t/period)
    x = np.cos(theta)
    y = np.sin(theta)*np.cos(incl)
    z = np.sin(theta)*np.sin(incl)
    lat = np.degrees(np.arcsin(z))
    lon = np.degrees(np.arctan2(y, x))

    #Earth rotation
    delta_lon = -np.degrees(omega_earth * (orbit_index*period + t))
    lon = ((lon + delta_lon + 180) % 360) - 180
    return lon, lat

def generate_swaths(n_days, samples_per_orbit=10):

# Generate swath polygons (1 km wide) for each orbit
    all_swaths = []
    n_orbits = int(np.floor(orbits_per_day * n_days))
    for i in range(n_orbits):
        t = np.linspace(0, period, samples_per_orbit)
        lon, lat = latlon_from_orbit(inc_deg, t, period, i)
        # Convert 1 km to ~0.009° latitude offset
        lat_offset = 0.009
        # Swath polygon: top edge (lat+), bottom edge (lat-)
        coords_upper = list(zip(lon, lat + lat_offset))
        coords_lower = list(zip(lon[::-1], (lat - lat_offset)[::-1]))
        swath_coords = coords_upper + coords_lower
        all_swaths.append(swath_coords)
    return all_swaths

def export_kml(swaths, filename="sso_swaths.kml"):

# Export swath polygons to KML
    kml = simplekml.Kml()
    for i, swath in enumerate(swaths):
        pol = kml.newpolygon(name=f"Orbit {i+1} Swath")
        pol.outerboundaryis = swath
        pol.style.polystyle.color = simplekml.Color.changealphaint(50, simplekml.Color.red)
        pol.style.linestyle.width = 1
    kml.save(filename)
    print(f"KML saved as {filename}")

# Run simulation
if __name__ == "__main__":
# Specify number of days
    days = 80
    swaths = generate_swaths(n_days=days)
    export_kml(swaths, "sso_474km_80days_swaths.kml")


Sun-sync inclination at 474 km: 97.29°
Period: 93.93 min, 15.33 orbits/day
KML saved as sso_474km_80days_swaths.kml
